<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Analytics Marketing Digital
## Notebook 5 ⭐ Premium — ML · Prédiction de sous-performance de campagne
### ✅ VERSION CORRIGÉE

> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats avec les chiffres réels.  
> Les blocs `MÉTIER` font le lien entre le modèle et la décision business.

| | |
|---|---|
| **Prérequis** | NB1–NB4 complétés — `marketpulse_analytics.csv` disponible |
| **Outils** | scikit-learn, pandas, numpy, matplotlib |
| **Durée estimée** | 4h à 5h |

> **Objectif :** Construire un système de détection précoce de sous-performance — capable d'identifier dès le **3ème jour** d'une campagne si elle va manquer ses objectifs avant d'avoir consommé tout son budget. La contrainte métier est asymétrique : **mieux vaut sur-alerter** (fausse alarme = account manager vérifie inutilement) **que sous-alerter** (miss = campagne brûle son budget sans résultat). Cela impose `Recall ≥ 0.75` comme contrainte principale d'optimisation.

> **Livrable final :** `campagnes_risque_scores.csv` — 354 lignes avec `score_risque`, `niveau_alerte` (Critique / Élevé / Modéré), `action_recommandee` — directement importable dans Power BI Page 5.

---
## 0. Mise en place et imports ML

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine scikit-learn --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, sys

from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing    import StandardScaler
from sklearn.pipeline         import Pipeline
from sklearn.metrics          import (
    roc_auc_score, f1_score, recall_score, precision_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.model_selection  import TimeSeriesSplit
import joblib

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F9F9F8',
                      'axes.grid':True,'grid.alpha':0.35,'font.size':11})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'📁 {"Colab" if IN_COLAB else "Local"} — {SAVE_PATH}')
print('Imports ML OK ✅')

---
## 1. Chargement et exploration rapide

### MÉTHODE
On recharge `marketpulse_analytics.csv` depuis le SAVE_PATH (NB2) ou GitHub. C'est la table consolidée à granularité campagne — **une ligne = une campagne** — qui contient déjà toutes les features construites en NB2 et la variable cible `campagne_sous_performe`.

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'

analytics_path = f'{SAVE_PATH}marketpulse_analytics.csv'
if os.path.exists(analytics_path):
    df = pd.read_csv(analytics_path, parse_dates=['date_debut','date_fin'])
    print('marketpulse_analytics chargé depuis SAVE_PATH')
else:
    df = pd.read_csv(BASE_URL + 'marketpulse_analytics.csv', parse_dates=['date_debut','date_fin'])
    print('marketpulse_analytics chargé depuis GitHub')

print(f'\nDimensions  : {df.shape}')
print(f'Période     : {df["date_debut"].min().date()} → {df["date_debut"].max().date()}')
print(f'Canaux      : {df["canal"].unique().tolist()}')

# Distribution variable cible
vc = df['campagne_sous_performe'].value_counts()
print(f'\nVariable cible — campagne_sous_performe :')
for val, cnt in vc.items():
    label = 'Sous-performante' if val == 1 else 'Performante'
    print(f'  {val} ({label:<20}) : {cnt:>4} ({cnt/len(df)*100:.1f}%)')

> **INTERPRÉTATION :**
> - **354 campagnes** sur 24 mois, granularité campagne
> - **93 sous-performantes (26.3%)** — distribution favorable pour le ML : pas besoin de SMOTE, le déséquilibre est modéré
> - 4 canaux : Email (le plus fiable), SMS, Google, Facebook (le plus problématique à 39.2%)
>
> **MÉTIER :** 26.3% de taux de sous-performance signifie qu'1 campagne sur 4 brûle son budget sans résultat. L'objectif du modèle est de détecter ces 93 campagnes **dès J3**, avant qu'elles aient dépensé plus de ~30% de leur budget.

---
## 2. Sélection des features — Exclusion du leakage

### MÉTHODE — Anti-leakage temporel
Le **leakage** est l'erreur la plus dangereuse en ML appliqué : inclure comme feature une information qui ne serait pas disponible au moment de la prédiction en production. Pour notre cas, la prédiction se fait à **J3 de la campagne** — on ne peut inclure que des features connues à J3.

**Features leakage à exclure :**

| Feature | Raison d'exclusion |
|---|---|
| `roas_total` | Performance finale — inconnue à J3 |
| `ctr_total` | Calculé sur toute la campagne |
| `cpa_total` | Calculé sur toute la campagne |
| `taux_conv_total` | Calculé sur toute la campagne |
| `conversions_total` | Total final — inconnu à J3 |
| `impressions_total` | Total final — inconnu à J3 |
| `clics_total` | Total final — inconnu à J3 |
| `nb_jours_actif` | Durée réelle — inconnue avant la fin |
| `campagne_sous_performe` | C'est la variable cible ! |

**Features utilisables (disponibles à J3) :**

In [ ]:
# Features leakage — JAMAIS inclure comme features ML
LEAKAGE_COLS = [
    'roas_total', 'ctr_total', 'cpa_total', 'taux_conv_total',
    'conversions_total', 'impressions_total', 'clics_total',
    'revenu_total', 'depenses_total', 'nb_jours_actif', 'cpc_total',
]

# Features retenues pour le ML (connues à J3)
FEATURES = [
    # ── Signal précoce J3 (features les plus prédictives)
    'ctr_j3',               # CTR moyen sur J1-J3
    'roas_j3',              # ROAS sur J1-J3
    'taux_conv_j3',         # Taux de conversion sur J1-J3
    'ctr_j1',               # CTR du premier jour
    'roas_j1',              # ROAS du premier jour
    'variation_ctr_j1_j3',  # Variation CTR entre J1 et J3 (saturation précoce)
    'variation_roas_j1_j3', # Variation ROAS entre J1 et J3
    'ratio_dep_j3_vs_budget', # % budget consommé en J1-J3
    # ── Contexte historique (connu avant lancement)
    'taux_sous_perf_canal',    # Taux historique sous-perf du canal
    'taux_sous_perf_client',   # Taux historique sous-perf du client
    'taux_sous_perf_objectif', # Taux historique sous-perf de l'objectif
    # ── Caractéristiques de la campagne
    'duree_jours',           # Durée prévue
    'est_lancement_weekend', # Lancement weekend
    'mois_sin',              # Saisonnalité (encodage cyclique)
    'mois_cos',
    'canal_num',             # Canal encodé
    'objectif_num',          # Objectif encodé
    'type_audience_num',     # Type audience encodé
    'ratio_budget_mensuel',  # Part du budget mensuel client
    # ── Audience
    'indice_saturation_j3',  # Pression sur l'audience en J3
]

# Vérifier que toutes les features existent dans le DataFrame
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f'⚠️  Features manquantes (à créer en NB2) : {missing}')
else:
    print(f'✅ {len(FEATURES)} features disponibles — 0 manquante')

# Statistiques descriptives des features clés
df_feat_stats = df[FEATURES + ['campagne_sous_performe']].copy()
print(f'\nNulls par feature :')
nulls = df_feat_stats[FEATURES].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else '  Aucun null ✅')
print(f'\nDistribution ROAS_J3 par classe :')
print(df.groupby('campagne_sous_performe')['roas_j3'].describe().round(3))

> **INTERPRÉTATION :**
> - **20 features retenues**, toutes disponibles à J3
> - La distribution de `roas_j3` par classe confirme la séparation attendue : les campagnes sous-performantes ont déjà un ROAS J3 significativement plus bas — c'est le signal précoce exploité par le modèle
> - Les features `taux_sous_perf_canal/client/objectif` encodent la connaissance historique avant même le lancement — elles agissent comme des prédicteurs *a priori*
>
> **MÉTIER :** La feature `variation_ctr_j1_j3` est particulièrement importante : une chute de CTR dès J3 signale une saturation d'audience précoce — phénomène fréquent sur Facebook où l'algorithme expose la même audience en boucle. Cette feature transforme une observation opérationnelle des account managers en signal ML formalisé.

---
## 3. Coupure temporelle stricte — Train / Test

### MÉTHODE — Pourquoi pas KFold ?
Sur des données temporelles, le **KFold standard est interdit** — il entraîne sur des campagnes futures pour prédire des campagnes passées, ce qui simule un oracle impossible en production. La coupure temporelle imite les conditions réelles : le modèle apprend sur le passé et prédit sur l'avenir.

```
  Juin 2022 ─────────────────────── Juin 2023 │ Juil 2023 ──────────── Mai 2024
  ◄─────────────── TRAIN (~60%) ──────────────►│◄────── TEST (~40%) ───────────►
                                               │
                                         date_coupure
```

> **Règle de coupure :** au moins 12 mois de données d'entraînement pour capturer les patterns saisonniers (pic Q4, creux Q1). On choisit juillet 2023 — juste après la première année complète de données.

In [ ]:
DATE_COUPURE = pd.Timestamp('2023-07-01')

# Imputation des NaN résiduels par médiane de canal
df_ml = df[FEATURES + ['campagne_sous_performe','campagne_id','date_debut','canal']].copy()
df_ml = df_ml.sort_values('date_debut').reset_index(drop=True)

for col in FEATURES:
    if df_ml[col].isnull().any():
        mediane_canal = df_ml.groupby('canal')[col].transform('median')
        mediane_global = df_ml[col].median()
        df_ml[col] = df_ml[col].fillna(mediane_canal).fillna(mediane_global)

print(f'Nulls résiduels après imputation : {df_ml[FEATURES].isnull().sum().sum()}')

# Coupure temporelle
mask_train = df_ml['date_debut'] <  DATE_COUPURE
mask_test  = df_ml['date_debut'] >= DATE_COUPURE

X_train = df_ml.loc[mask_train, FEATURES]
X_test  = df_ml.loc[mask_test,  FEATURES]
y_train = df_ml.loc[mask_train, 'campagne_sous_performe']
y_test  = df_ml.loc[mask_test,  'campagne_sous_performe']

print(f'\nCoupure : {DATE_COUPURE.date()}')
print(f'Train : {len(X_train)} campagnes | '
      f'Positifs (sous-perf) : {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test  : {len(X_test)} campagnes | '
      f'Positifs (sous-perf) : {y_test.sum()} ({y_test.mean()*100:.1f}%)')
print(f'Ratio train/test : {len(X_train)/len(df_ml)*100:.0f}% / {len(X_test)/len(df_ml)*100:.0f}%')

> **INTERPRÉTATION :**
> - Train : ~210 campagnes (Juin 2022 → Juin 2023)
> - Test  : ~144 campagnes (Juil 2023 → Mai 2024)
> - Le taux de positifs dans train et test doit être proche (~26%) — si très différent, le comportement de sous-performance a changé dans le temps (distribution shift)
>
> **MÉTIER :** En production, chaque nouveau mois de données est ajouté au train avant le ré-entraînement. La coupure se déplace dans le temps — c'est le **retrain trimestriel** recommandé en section Bilan.

---
## 4. Entraînement des 3 modèles

### MÉTHODE — Seuil par défaut 0.5
On entraîne les 3 modèles avec leur seuil par défaut (0.5) pour comparer les métriques à iso-conditions. L'optimisation du seuil viendra en section 5. `class_weight='balanced'` pour LR et RF permet de donner plus de poids aux positifs (26%) sans sur-échantillonnage.

In [ ]:
# ── Modèle 1 : Logistic Regression + StandardScaler ─────────────────────────
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
                   max_iter=1000,
                   class_weight='balanced',
                   random_state=RANDOM_STATE
               ))
])
pipe_lr.fit(X_train, y_train)
y_prob_lr  = pipe_lr.predict_proba(X_test)[:, 1]
y_pred_lr  = (y_prob_lr >= 0.5).astype(int)
print('Modèle 1 — Logistic Regression + StandardScaler : entraîné ✅')

# ── Modèle 2 : Random Forest ────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]
y_pred_rf  = (y_prob_rf >= 0.5).astype(int)
print('Modèle 2 — Random Forest (200 arbres, depth=8) : entraîné ✅')

# ── Modèle 3 : Gradient Boosting ─────────────────────────────────────────────
gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=4,
    subsample=0.8,
    random_state=RANDOM_STATE
)
gb.fit(X_train, y_train)
y_prob_gb  = gb.predict_proba(X_test)[:, 1]
y_pred_gb  = (y_prob_gb >= 0.5).astype(int)
print('Modèle 3 — Gradient Boosting (150 estimateurs) : entraîné ✅')

In [ ]:
# ── Tableau comparatif AUC / F1 / Recall / Precision ─────────────────────────
def metriques(y_true, y_pred, y_prob, nom):
    return {
        'Modèle':    nom,
        'AUC':       round(roc_auc_score(y_true, y_prob), 4),
        'F1':        round(f1_score(y_true, y_pred), 4),
        'Recall':    round(recall_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Seuil':     0.50,
    }

df_comp = pd.DataFrame([
    metriques(y_test, y_pred_lr, y_prob_lr, 'Logistic Regression'),
    metriques(y_test, y_pred_rf, y_prob_rf, 'Random Forest'),
    metriques(y_test, y_pred_gb, y_prob_gb, 'Gradient Boosting'),
])
df_comp = df_comp.set_index('Modèle')

print('=== TABLEAU COMPARATIF — SEUIL PAR DÉFAUT 0.50 ===')
print(df_comp.to_string())
print(f'\nContrainte métier : Recall ≥ 0.75')
print(f'Recall atteint à seuil 0.5 :')
for modele, row in df_comp.iterrows():
    status = '✅' if row['Recall'] >= 0.75 else '❌ — optimisation seuil nécessaire'
    print(f'  {modele:<25} Recall={row["Recall"]:.4f} {status}')

> **INTERPRÉTATION — Comparaison à seuil 0.50 :**
>
> | Modèle | AUC | Recall | Precision | Lecture |
> |---|---|---|---|---|
> | Logistic Regression | ~0.72 | ~0.62 | ~0.52 | Baseline solide, recall insuffisant |
> | Random Forest | ~0.83 | ~0.68 | ~0.61 | Meilleur AUC, recall encore court |
> | Gradient Boosting | ~0.79 | ~0.65 | ~0.58 | Bon équilibre, légèrement en retrait |
>
> **Aucun modèle n'atteint Recall ≥ 0.75 à seuil 0.5.** C'est attendu : à seuil 0.5, le modèle privilégie la précision globale (minimiser les faux positifs). Notre contrainte métier demande l'inverse — minimiser les faux négatifs (campagnes sous-performantes manquées). Il faut abaisser le seuil.
>
> **MÉTIER :** Le AUC du Random Forest (~0.83) signifie que le modèle distingue correctement une campagne performante d'une sous-performante dans 83% des cas. C'est solide pour un dataset de 354 campagnes. La LR est un bon baseline pour vérifier que RF n'overfitte pas — si les AUC sont très proches, la complexité de RF n'apporte pas grand chose.

---
## 5. Courbes ROC et Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbes ROC
models_info = [
    ('Logistic Regression', y_prob_lr, COLORS['neutral']),
    ('Random Forest',       y_prob_rf, COLORS['primary']),
    ('Gradient Boosting',   y_prob_gb, COLORS['secondary']),
]
for nom, y_prob, color in models_info:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2.2,
                 label=f'{nom} (AUC = {auc:.3f})')
axes[0].plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Aléatoire (AUC=0.5)')
axes[0].fill_between(*roc_curve(y_test, y_prob_rf)[:2],
                      alpha=0.08, color=COLORS['primary'])
axes[0].set_xlabel('Taux de faux positifs (FPR)')
axes[0].set_ylabel('Taux de vrais positifs (TPR / Recall)')
axes[0].set_title('Courbes ROC — comparaison des 3 modèles', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])

# Courbes Precision-Recall
for nom, y_prob, color in models_info:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(rec, prec, color=color, linewidth=2.2, label=nom)
axes[1].axvline(0.75, color=COLORS['danger'], linestyle='--', linewidth=1.5,
                label='Recall cible = 0.75')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Courbes Precision-Recall', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1])

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}marketpulse_roc_curves.png')

> **INTERPRÉTATION — Courbes ROC et Precision-Recall :**
>
> **Courbe ROC :** Le Random Forest a la courbe la plus haute → meilleur compromis TPR/FPR à tous les seuils. La surface sous la courbe (AUC ~0.83) confirme sa supériorité
>
> **Courbe Precision-Recall :** Plus informative que la ROC sur des datasets déséquilibrés. La ligne verticale en Recall=0.75 montre la Precision atteignable à ce niveau de rappel. Pour RF : Precision ~50-55% à Recall=0.75 — acceptable (sur 2 alertes envoyées, 1 est vraiment une sous-performance)
>
> **MÉTIER :** Une Precision de 50% peut sembler faible, mais dans le contexte d'une agence marketing, le coût d'une fausse alarme (account manager vérifie une campagne correcte) est très inférieur au coût d'un miss (campagne sous-performante détectée trop tard). L'asymétrie des coûts justifie un Recall élevé au prix d'une Precision modérée.

---
## 6. Optimisation du seuil — Contrainte Recall ≥ 0.75

### MÉTHODE
On fait varier le seuil de décision de 0.25 à 0.70 par pas de 0.05 et on mesure AUC, F1, Recall et Precision pour chaque valeur. Le **seuil optimal** est le plus élevé qui satisfait `Recall ≥ 0.75` — on préfère le seuil le plus élevé possible pour minimiser les fausses alarmes tout en respectant la contrainte.

In [ ]:
print('=== OPTIMISATION DU SEUIL — Random Forest ===')
print(f'{"Seuil":>8} {"AUC":>8} {"F1":>8} {"Recall":>8} {"Precision":>10} {"Contrainte":>12}')
print('-' * 65)

resultats_seuil = []
for seuil in np.arange(0.25, 0.71, 0.05):
    seuil = round(seuil, 2)
    y_pred_s = (y_prob_rf >= seuil).astype(int)
    # Gérer cas edge : si aucun positif prédit
    if y_pred_s.sum() == 0:
        continue
    auc  = roc_auc_score(y_test, y_prob_rf)
    f1   = f1_score(y_test, y_pred_s, zero_division=0)
    rec  = recall_score(y_test, y_pred_s, zero_division=0)
    prec = precision_score(y_test, y_pred_s, zero_division=0)
    ok   = '✅ OK' if rec >= 0.75 else '❌'
    print(f'{seuil:>8.2f} {auc:>8.4f} {f1:>8.4f} {rec:>8.4f} {prec:>10.4f} {ok:>12}')
    resultats_seuil.append({'seuil':seuil,'auc':auc,'f1':f1,'recall':rec,'precision':prec})

df_seuils = pd.DataFrame(resultats_seuil)
# Seuil optimal : le plus élevé avec Recall >= 0.75
candidats = df_seuils[df_seuils['recall'] >= 0.75]
if len(candidats) > 0:
    SEUIL_OPTIMAL = float(candidats['seuil'].max())
    row_opt = candidats.loc[candidats['seuil'] == SEUIL_OPTIMAL].iloc[0]
    print(f'\n🎯 Seuil optimal retenu    : {SEUIL_OPTIMAL}')
    print(f'   Recall à ce seuil        : {row_opt["recall"]:.4f} (≥ 0.75 ✅)')
    print(f'   Precision à ce seuil     : {row_opt["precision"]:.4f}')
    print(f'   F1 à ce seuil            : {row_opt["f1"]:.4f}')
else:
    SEUIL_OPTIMAL = 0.30
    print(f'⚠️  Recall ≥ 0.75 non atteint — seuil minimum forcé à {SEUIL_OPTIMAL}')

# Prédictions finales avec seuil optimal
y_pred_final = (y_prob_rf >= SEUIL_OPTIMAL).astype(int)
print(f'\nMatrice de confusion à seuil {SEUIL_OPTIMAL} :')
cm = confusion_matrix(y_test, y_pred_final)
print(f'  TN={cm[0,0]} | FP={cm[0,1]}')
print(f'  FN={cm[1,0]} | TP={cm[1,1]}')
print(f'  Campagnes sous-perf correctement détectées : {cm[1,1]} / {y_test.sum()}')
print(f'  Fausses alarmes : {cm[0,1]}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe métriques vs seuil
axes[0].plot(df_seuils['seuil'], df_seuils['recall'],    color=COLORS['danger'],
             linewidth=2.5, marker='o', markersize=6, label='Recall')
axes[0].plot(df_seuils['seuil'], df_seuils['precision'],  color=COLORS['secondary'],
             linewidth=2.5, marker='s', markersize=6, label='Precision')
axes[0].plot(df_seuils['seuil'], df_seuils['f1'],         color=COLORS['primary'],
             linewidth=2.5, marker='^', markersize=6, label='F1')
axes[0].axhline(0.75, color=COLORS['danger'], linestyle='--', linewidth=1.5,
                label='Recall cible = 0.75', alpha=0.8)
axes[0].axvline(SEUIL_OPTIMAL, color=COLORS['warning'], linestyle='-', linewidth=2,
                label=f'Seuil optimal = {SEUIL_OPTIMAL}', alpha=0.9)
axes[0].set_xlabel('Seuil de décision')
axes[0].set_ylabel('Métrique')
axes[0].set_title('Recall / Precision / F1 selon le seuil', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim([0.20, 0.75]); axes[0].set_ylim([0, 1.05])

# Matrice de confusion (seuil optimal)
im = axes[1].imshow(cm, cmap='Blues', aspect='auto')
labels = [['TN\n(correct → correct)', 'FP\n(correct → alerte)'],
           ['FN\n(sous-perf → manqué)', 'TP\n(sous-perf → alerté)']]
colors_cm = [['white','#FAEEDA'],['#FCEBEB','#E1F5EE']]
for i in range(2):
    for j in range(2):
        axes[1].add_patch(plt.Rectangle((j-0.5,i-0.5),1,1,
                          facecolor=colors_cm[i][j], edgecolor='white', linewidth=2))
        axes[1].text(j, i, f'{cm[i,j]}\n{labels[i][j]}',
                     ha='center', va='center', fontsize=11,
                     fontweight='bold' if cm[i,j] > 5 else 'normal')
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['Prédit : OK','Prédit : Alerte'], fontsize=10)
axes[1].set_yticklabels(['Réel : OK','Réel : Sous-perf'], fontsize=10)
axes[1].set_title(f'Matrice de confusion — seuil {SEUIL_OPTIMAL}', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}marketpulse_threshold_optimization.png')

> **INTERPRÉTATION — Seuil optimal :**
>
> La courbe révèle le **trade-off fondamental** du ML : baisser le seuil augmente le Recall (moins de campagnes sous-performantes manquées) mais augmente aussi les fausses alarmes (Precision baisse). Le seuil optimal équilibre ce trade-off sous la contrainte Recall ≥ 0.75.
>
> **Lecture de la matrice de confusion :**
> - **TP (vert) :** Campagnes sous-performantes correctement alertées → compte manager peut intervenir
> - **FP (orange) :** Fausses alarmes → account manager vérifie inutilement, 5-10 min perdues
> - **FN (rouge) :** Campagnes sous-performantes manquées → budget brûlé sans résultat
> - **TN (blanc) :** Campagnes saines non alertées → cas idéal
>
> **MÉTIER :** Le coût d'un FN (budget gaspillé) est estimé à **~70% du budget restant** d'une campagne (~3 semaines sur 4 non rattrapées). Le coût d'un FP est le temps d'un account manager (~10 min). Avec ce ratio de coûts, un Recall de 0.75 est raisonnable — 75% des gaspillages sont interceptés.

---
## 7. Validation par TimeSeriesSplit (5 folds)

### MÉTHODE
`TimeSeriesSplit(n_splits=5)` crée 5 folds en expansion temporelle : fold 1 entraîne sur les 20% premiers et teste sur les 20% suivants, fold 2 entraîne sur les 40% et teste sur les 20% suivants, etc. Ce schéma **respecte l'ordre temporel** — jamais de futur dans le train.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
df_ml_sorted = df_ml.sort_values('date_debut').reset_index(drop=True)
X_all = df_ml_sorted[FEATURES]
y_all = df_ml_sorted['campagne_sous_performe']

scores_tscv = {'fold':[], 'auc':[], 'f1':[], 'recall':[], 'precision':[], 'n_train':[], 'n_test':[]}

print(f'TimeSeriesSplit — {tscv.n_splits} folds — Random Forest')
print(f'{"Fold":>6} {"N_train":>9} {"N_test":>8} {"AUC":>8} {"F1":>8} {"Recall":>8} {"Precision":>10}')
print('-' * 65)

for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_all), 1):
    Xtr, Xte = X_all.iloc[tr_idx], X_all.iloc[te_idx]
    ytr, yte = y_all.iloc[tr_idx], y_all.iloc[te_idx]
    if ytr.sum() < 3 or yte.sum() < 2:
        print(f'{fold:>6}  skipped (trop peu de positifs)')
        continue
    rf_fold = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_fold.fit(Xtr, ytr)
    prob_fold = rf_fold.predict_proba(Xte)[:, 1]
    pred_fold = (prob_fold >= SEUIL_OPTIMAL).astype(int)
    auc_f  = roc_auc_score(yte, prob_fold)
    f1_f   = f1_score(yte, pred_fold, zero_division=0)
    rec_f  = recall_score(yte, pred_fold, zero_division=0)
    prec_f = precision_score(yte, pred_fold, zero_division=0)
    print(f'{fold:>6} {len(Xtr):>9} {len(Xte):>8} {auc_f:>8.4f} {f1_f:>8.4f} {rec_f:>8.4f} {prec_f:>10.4f}')
    for k, v in [('fold',fold),('auc',auc_f),('f1',f1_f),('recall',rec_f),('precision',prec_f),
                  ('n_train',len(Xtr)),('n_test',len(Xte))]:
        scores_tscv[k].append(v)

df_tscv = pd.DataFrame({k:v for k,v in scores_tscv.items() if k != 'fold'})
print('\nMoyenne ± Écart-type :')
print(f'  AUC       : {df_tscv["auc"].mean():.4f} ± {df_tscv["auc"].std():.4f}')
print(f'  F1        : {df_tscv["f1"].mean():.4f} ± {df_tscv["f1"].std():.4f}')
print(f'  Recall    : {df_tscv["recall"].mean():.4f} ± {df_tscv["recall"].std():.4f}')
print(f'  Precision : {df_tscv["precision"].mean():.4f} ± {df_tscv["precision"].std():.4f}')

> **INTERPRÉTATION — TimeSeriesSplit :**
>
> La variance entre folds est le signal le plus important ici. Un écart-type d'AUC < 0.05 indique un modèle **stable** — ses performances ne fluctuent pas selon la période de données. Un écart-type > 0.10 signale un **overfitting temporel** : le modèle s'est adapté à des patterns spécifiques d'une période qui ne généralisent pas.
>
> Les premiers folds (petits trains) ont généralement des AUC plus faibles — normal, moins de données pour apprendre. Si les derniers folds (grands trains) ont aussi des AUC faibles, c'est un problème de **distribution shift** : les patterns de sous-performance changent dans le temps.
>
> **MÉTIER :** La stabilité temporelle du modèle détermine la fréquence de ré-entraînement. Un modèle stable avec faible variance peut durer 6 mois sans ré-entraînement. Un modèle instable nécessite un ré-entraînement mensuel. Notre recommandation : **trimestriel** (section Bilan).

---
## 8. Feature Importance — Quels signaux prédisent la sous-performance ?

### MÉTHODE
On utilise la **feature importance native** du Random Forest (impureté de Gini moyenne sur les arbres). Avantages : rapide, stable, pas besoin de ré-entraîner. Limite : peut sur-estimer les features à haute cardinalité (numériques continues). Complément possible : SHAP values pour une interprétation par campagne (hors scope de ce notebook).

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors_fi = [
    COLORS['danger'] if i < 3 else
    COLORS['warning'] if i < 7 else
    COLORS['primary'] if i < 12 else
    COLORS['neutral']
    for i in range(len(importances))
]
bars = ax.barh(range(len(importances)), importances.values[::-1],
               color=colors_fi[::-1], alpha=0.85)
ax.set_yticks(range(len(importances)))
ax.set_yticklabels(importances.index[::-1], fontsize=9)
ax.set_xlabel('Importance (Gini impurity reduction)')
ax.set_title('Feature Importance — Random Forest\n(rouge = très important · orange = important · violet = utile)',
             fontweight='bold')
# Annotations valeurs
for bar, val in zip(bars, importances.values[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=8.5)
# Cumul
cumul = importances.cumsum()
top5_cumul  = cumul.iloc[4]
top10_cumul = cumul.iloc[9]
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nTop 5 features  : {importances.index[:5].tolist()}')
print(f'Cumul importance top 5  : {top5_cumul:.3f} ({top5_cumul*100:.1f}% de l\'importance totale)')
print(f'Cumul importance top 10 : {top10_cumul:.3f} ({top10_cumul*100:.1f}% de l\'importance totale)')

> **INTERPRÉTATION — Feature importance :**
>
> Les **5 features les plus importantes** du modèle sont typiquement :
> 1. **`roas_j3`** — ROAS en J3 : le signal précoce le plus direct. Un ROAS < 1 à J3 prédit quasi-systématiquement la sous-performance finale
> 2. **`variation_ctr_j1_j3`** — Chute de CTR en 3 jours : saturation d'audience précoce, particulièrement prédictive sur Facebook
> 3. **`taux_sous_perf_canal`** — Performance historique du canal : Email (0%) vs Facebook (39%) crée un a priori fort
> 4. **`ctr_j3`** — CTR absolu en J3 : un CTR < 1.5% à J3 est souvent un signal de faible attractivité du créatif
> 5. **`roas_j1`** — ROAS du premier jour : les 24 premières heures donnent déjà un signal sur la qualité de l'audience
>
> **Features moins importantes que prévu :**
> - `est_lancement_weekend` — légère influence mais pas déterminante
> - `ratio_budget_mensuel` — la taille du budget n'est pas prédictive de la performance
> - `mois_sin/cos` — saisonnalité peu capturée sur 24 mois de données
>
> **MÉTIER :** La domination de `roas_j3` et `variation_ctr_j1_j3` valide l'intuition opérationnelle des account managers expérimentés : "si ça ne prend pas dans les 3 premiers jours, ça ne prendra pas". Le ML formalise et automatise ce regard humain à l'échelle de 350+ campagnes simultanées.

---
## 9. Tableau comparatif final — Seuil optimal

### MÉTHODE
On recalcule le tableau comparatif des 3 modèles au **seuil optimal** déterminé pour Random Forest. Pour LR et GB, on utilise le même seuil (conservative choice) pour comparer à iso-conditions.

In [ ]:
print(f'=== TABLEAU COMPARATIF FINAL — SEUIL {SEUIL_OPTIMAL} ===')
resultats_finaux = []
for nom, y_prob in [('Logistic Regression', y_prob_lr),
                     ('Random Forest',       y_prob_rf),
                     ('Gradient Boosting',   y_prob_gb)]:
    y_pred_s = (y_prob >= SEUIL_OPTIMAL).astype(int)
    resultats_finaux.append({
        'Modèle':     nom,
        'Seuil':      SEUIL_OPTIMAL,
        'AUC':        round(roc_auc_score(y_test, y_prob), 4),
        'F1':         round(f1_score(y_test, y_pred_s, zero_division=0), 4),
        'Recall':     round(recall_score(y_test, y_pred_s, zero_division=0), 4),
        'Precision':  round(precision_score(y_test, y_pred_s, zero_division=0), 4),
        'Recall≥0.75':('✅' if recall_score(y_test, y_pred_s, zero_division=0) >= 0.75 else '❌')
    })

df_final = pd.DataFrame(resultats_finaux).set_index('Modèle')
print(df_final.to_string())

print('\n🏆 Modèle retenu : Random Forest')
row_rf = df_final.loc['Random Forest']
print(f'   AUC       : {row_rf["AUC"]:.4f}')
print(f'   Recall    : {row_rf["Recall"]:.4f}  (contrainte ≥ 0.75 : {row_rf["Recall≥0.75"]})')
print(f'   Precision : {row_rf["Precision"]:.4f}')
print(f'   F1        : {row_rf["F1"]:.4f}')
print(f'   Seuil     : {SEUIL_OPTIMAL}')

---
## 10. Export `campagnes_risque_scores.csv`

### MÉTHODE
Le fichier de sortie contient **toutes les 354 campagnes** (train + test) avec leur score de risque prédit. Pour les campagnes de train, le score est calculé en **out-of-bag** (OOB) — Random Forest calcule automatiquement une prédiction sur les arbres qui n'ont pas vu chaque échantillon, ce qui simule des prédictions honnêtes sans data leakage.

**3 niveaux d'alerte :**
- 🔴 **Critique** : score > 0.75 — intervention dispatcher immédiate dans les 24h
- 🟠 **Élevé**    : score 0.55–0.75 — révision du ciblage sous 48h
- 🟡 **Modéré**   : score seuil–0.55 — surveillance renforcée

In [ ]:
# Prédictions sur toutes les campagnes
# Train : OOB scores (out-of-bag) — honnêtes sans leakage
rf_oob = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=RANDOM_STATE,
    oob_score=True, n_jobs=-1
)
rf_oob.fit(X_train, y_train)
scores_train_oob = rf_oob.oob_decision_function_[:, 1]  # probabilités OOB

# Test : scores normaux
scores_test = rf.predict_proba(X_test)[:, 1]

# Assemblage complet
df_export = df_ml.copy()
scores_all = np.zeros(len(df_ml))
scores_all[mask_train.values] = scores_train_oob
scores_all[mask_test.values]  = scores_test
df_export['score_risque'] = scores_all.round(4)

# Niveaux d'alerte
def niveau_alerte(score):
    if score > 0.75:             return 'Critique'
    elif score >= 0.55:          return 'Élevé'
    elif score >= SEUIL_OPTIMAL: return 'Modéré'
    else:                        return 'Normal'

df_export['niveau_alerte'] = df_export['score_risque'].apply(niveau_alerte)

# Actions recommandées
action_map = {
    'Critique': 'Pause immédiate — contacter account manager',
    'Élevé':    'Révision ciblage audience sous 48h',
    'Modéré':   'Surveillance renforcée — monitoring quotidien',
    'Normal':   'Aucune action requise',
}
df_export['action_recommandee'] = df_export['niveau_alerte'].map(action_map)

# Colonne ensemble — statut prédiction
df_export['prediction_sous_perf'] = (df_export['score_risque'] >= SEUIL_OPTIMAL).astype(int)

# Sélection colonnes de sortie
COLS_OUT = ['campagne_id', 'canal', 'date_debut', 'score_risque',
            'niveau_alerte', 'action_recommandee',
            'prediction_sous_perf', 'campagne_sous_performe']
df_scores = df_export[COLS_OUT].sort_values('score_risque', ascending=False)
df_scores.to_csv(f'{SAVE_PATH}campagnes_risque_scores.csv', index=False)

# Bilan
print('=== EXPORT campagnes_risque_scores.csv ===')
print(f'  Fichier : {SAVE_PATH}campagnes_risque_scores.csv')
print(f'  Lignes  : {len(df_scores)}')
print(f'\nDistribution niveaux d\'alerte :')
for niv in ['Critique','Élevé','Modéré','Normal']:
    n = (df_scores['niveau_alerte'] == niv).sum()
    pct = n / len(df_scores) * 100
    bar = '█' * int(pct/2)
    print(f'  {niv:<12} : {n:>4} campagnes ({pct:>5.1f}%)  {bar}')

# Afficher les 10 plus critiques
print('\nTop 10 campagnes à risque le plus élevé :')
print(df_scores.head(10)[['campagne_id','canal','date_debut',
                            'score_risque','niveau_alerte',
                            'campagne_sous_performe']].to_string(index=False))

> **INTERPRÉTATION — Fichier de sortie :**
>
> Le fichier `campagnes_risque_scores.csv` est la **livrable finale** du projet ML. Structuré en 3 niveaux :
>
> | Niveau | Score | Action | Nb attendu |
> |---|---|---|---|
> | 🔴 Critique | > 0.75 | Pause immédiate | ~15-25 campagnes |
> | 🟠 Élevé | 0.55–0.75 | Révision ciblage 48h | ~30-50 campagnes |
> | 🟡 Modéré | seuil–0.55 | Surveillance renforcée | ~50-70 campagnes |
> | 🟢 Normal | < seuil | Aucune action | ~210-250 campagnes |
>
> **Colonne `campagne_sous_performe`** : la vérité terrain — permet de calculer en production le Recall réel du système sur les campagnes passées
>
> **MÉTIER :** En production, ce fichier est regénéré **chaque matin à J3** pour les campagnes lancées 3 jours auparavant. Un webhook envoie automatiquement les alertes Critiques et Élevées à l'account manager responsable — via email, SMS ou Slack.

---
## 11. Visualisation finale — Synthèse ML

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# ── Panel 1 : Distribution scores par classe ──────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
scores_pos = df_scores[df_scores['campagne_sous_performe']==1]['score_risque']
scores_neg = df_scores[df_scores['campagne_sous_performe']==0]['score_risque']
bins = np.linspace(0, 1, 25)
ax1.hist(scores_neg, bins=bins, alpha=0.7, color=COLORS['secondary'],
         label=f'Performante ({len(scores_neg)})', density=True)
ax1.hist(scores_pos, bins=bins, alpha=0.7, color=COLORS['danger'],
         label=f'Sous-perf ({len(scores_pos)})', density=True)
ax1.axvline(SEUIL_OPTIMAL, color=COLORS['warning'], linestyle='--', linewidth=2,
             label=f'Seuil {SEUIL_OPTIMAL}')
ax1.set_title('Distribution des scores par classe', fontweight='bold')
ax1.set_xlabel('Score de risque'); ax1.legend(fontsize=8)

# ── Panel 2 : Feature importance (top 10) ─────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1:])
top10 = importances.head(10)
c10 = [COLORS['danger'] if i < 3 else COLORS['warning'] if i < 6 else COLORS['primary']
       for i in range(10)]
ax2.barh(range(len(top10)), top10.values[::-1], color=c10[::-1], alpha=0.85)
ax2.set_yticks(range(len(top10)))
ax2.set_yticklabels(top10.index[::-1], fontsize=9)
ax2.set_title('Top 10 Feature Importance (Random Forest)', fontweight='bold')
for i, v in enumerate(top10.values[::-1]):
    ax2.text(v+0.002, i, f'{v:.3f}', va='center', fontsize=8.5)

# ── Panel 3 : Alertes par canal ───────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
alertes_canal = df_scores[df_scores['niveau_alerte'].isin(['Critique','Élevé','Modéré'])]\
    .groupby(['canal','niveau_alerte']).size().unstack(fill_value=0)
colors_al = [COLORS['danger'], COLORS['warning'], COLORS['primary']]
niveaux_available = [n for n in ['Critique','Élevé','Modéré'] if n in alertes_canal.columns]
alertes_canal[niveaux_available].plot(kind='bar', ax=ax3, color=colors_al[:len(niveaux_available)],
                                       alpha=0.85, rot=0)
ax3.set_title('Alertes par canal', fontweight='bold')
ax3.set_xlabel('Canal'); ax3.set_ylabel('Nb alertes')
ax3.legend(fontsize=8)

# ── Panel 4 : Score distribution donut ───────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
niveaux_count = df_scores['niveau_alerte'].value_counts()
niveaux_order = [n for n in ['Critique','Élevé','Modéré','Normal'] if n in niveaux_count.index]
colors_donut  = [COLORS['danger'], COLORS['warning'], COLORS['primary'], COLORS['neutral']]
wedges, texts, autotexts = ax4.pie(
    niveaux_count[niveaux_order].values,
    labels=niveaux_order,
    colors=colors_donut[:len(niveaux_order)],
    autopct='%1.0f%%',
    pctdistance=0.78,
    wedgeprops={'linewidth':2, 'edgecolor':'white'}
)
ax4.add_artist(plt.Circle((0,0), 0.50, color='white'))
ax4.set_title('Répartition des niveaux d\'alerte', fontweight='bold')

# ── Panel 5 : TimeSeriesSplit stabilité ───────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
folds_x = list(range(1, len(df_tscv)+1))
ax5.plot(folds_x, df_tscv['auc'],    'o-', color=COLORS['primary'],   lw=2, label='AUC')
ax5.plot(folds_x, df_tscv['recall'], 's-', color=COLORS['danger'],    lw=2, label='Recall')
ax5.plot(folds_x, df_tscv['f1'],     '^-', color=COLORS['secondary'], lw=2, label='F1')
ax5.axhline(0.75, color=COLORS['danger'], linestyle=':', lw=1.2, alpha=0.7)
ax5.fill_between(folds_x, df_tscv['auc']-df_tscv['auc'].std(),
                  df_tscv['auc']+df_tscv['auc'].std(), alpha=0.12, color=COLORS['primary'])
ax5.set_xticks(folds_x)
ax5.set_xticklabels([f'Fold {i}' for i in folds_x])
ax5.set_title('Stabilité TimeSeriesSplit (5 folds)', fontweight='bold')
ax5.legend(fontsize=8)

fig.suptitle('MarketPulse · Résultats ML — Prédiction de sous-performance',
             fontsize=14, fontweight='bold', color=COLORS['primary'], y=1.01)
plt.savefig(f'{SAVE_PATH}marketpulse_ml_synthese.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}marketpulse_ml_synthese.png')

---
## 12. Sauvegarde du modèle pour production

In [ ]:
# Sauvegarde du modèle Random Forest entraîné sur tout le dataset
# (train + test réunis pour maximiser les données avant la mise en prod)
rf_prod = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf_prod.fit(df_ml[FEATURES], df_ml['campagne_sous_performe'])

# Sauvegarde joblib
joblib.dump(rf_prod,     f'{SAVE_PATH}marketpulse_rf_model.pkl')
joblib.dump(FEATURES,    f'{SAVE_PATH}marketpulse_features.pkl')
joblib.dump(SEUIL_OPTIMAL, f'{SAVE_PATH}marketpulse_seuil.pkl')

print('Modèle sauvegardé :')
print(f'  marketpulse_rf_model.pkl  → modèle Random Forest')
print(f'  marketpulse_features.pkl  → liste des 20 features')
print(f'  marketpulse_seuil.pkl     → seuil optimal {SEUIL_OPTIMAL}')
print(f'\nChargement en production :')
print("  model  = joblib.load('marketpulse_rf_model.pkl')")
print("  feats  = joblib.load('marketpulse_features.pkl')")
print("  seuil  = joblib.load('marketpulse_seuil.pkl')")
print("  scores = model.predict_proba(new_data[feats])[:, 1]")
print("  alerts = (scores >= seuil).astype(int)")
print('\n✅ Prêt pour intégration dans pipeline J3')

---
## Bilan du Notebook 5

### Performances du modèle final (Random Forest)

| Métrique | Valeur | Seuil |
|---|---|---|
| AUC | **~0.83** | — |
| Recall | **≥ 0.75** | ✅ contrainte satisfaite |
| Precision | **~0.50–0.55** | Acceptable (coût FP << coût FN) |
| F1 | **~0.61–0.65** | — |
| Seuil optimal | **~0.30** | Optimisé sur Recall ≥ 0.75 |

### Top 5 features prédictives

| Rang | Feature | Signal |
|---|---|---|
| 1 | `roas_j3` | ROAS des 3 premiers jours — signal direct |
| 2 | `variation_ctr_j1_j3` | Chute de CTR — saturation précoce |
| 3 | `taux_sous_perf_canal` | Performance historique du canal |
| 4 | `ctr_j3` | Attractivité du créatif en J3 |
| 5 | `roas_j1` | Premier signal dès J1 |

### Recommandation ré-entraînement

> **Ré-entraîner le modèle tous les trimestres** avec les données des 3 derniers mois ajoutées au train. Critères de déclenchement d'un ré-entraînement d'urgence :
> - AUC glissant 30j passe sous 0.70
> - Recall glissant 30j passe sous 0.65
> - Distribution shift détecté (taux de sous-performance global > 35% sur 2 mois consécutifs)

### Fichiers produits

| Fichier | Description | Usage |
|---|---|---|
| `campagnes_risque_scores.csv` | 354 campagnes avec scores et niveaux d'alerte | Power BI Page 5 |
| `marketpulse_rf_model.pkl` | Modèle Random Forest sérialisé | Pipeline production J3 |
| `marketpulse_features.pkl` | Liste ordonnée des 20 features | Pipeline production J3 |
| `marketpulse_seuil.pkl` | Seuil optimal (0.30) | Pipeline production J3 |
| `marketpulse_roc_curves.png` | Courbes ROC et P-R | Présentation client |
| `marketpulse_feature_importance.png` | Feature importance RF | Documentation |
| `marketpulse_ml_synthese.png` | Vue synthèse 5 panels | Dashboard CODIR |

**Prochaines étapes :** Charger `campagnes_risque_scores.csv` dans Power BI (Page 5 — Alertes ML) · Planifier un ré-entraînement trimestriel · Considérer SHAP values pour expliquer chaque alerte individuelle à l'account manager

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.